In [1]:
!pip install -q transformers torch

In [2]:
# Task 1: Model Loading
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading the transformer model... (This might take a minute)")
model_name = "microsoft/DialoGPT-medium" # DialoGPT is optimized for conversational responses
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded successfully!\n")

def start_chatbot():
    """
    Task 4: Continuous Conversation
    Maintains the chat loop and conversation history.
    """
    print("-" * 50)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("(Type 'exit' or 'quit' to stop the conversation)")
    print("-" * 50)

    chat_history_ids = None # This will store the context of the conversation
    step = 0

    while True:
        # Task 2: User Input Handling
        user_input = input("User: ")

        # Task 5: Exit Condition
        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Have a great day!")
            break

        if user_input.strip() == "":
            continue

        # Encode the new user input, add the eos_token (end of string) and return a tensor in Pytorch
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        # Append the new user input tokens to the chat history (if it's not the first turn)
        if step > 0:
            bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
        else:
            bot_input_ids = new_user_input_ids

        # Task 3: Response Generation
        # Generate a response while limiting the total chat history to 1000 tokens
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,       # Prevents the bot from repeating the same phrases
            do_sample=True,               # Enables diverse text generation
            top_k=50,
            top_p=0.95,
            temperature=0.7               # Makes responses slightly more creative/natural
        )

        # Extract only the bot's newly generated response from the history
        step_response = chat_history_ids[:, bot_input_ids.shape[-1]:]

        # Decode the tokens back into text
        bot_response = tokenizer.decode(step_response[0], skip_special_tokens=True)

        print(f"Chatbot: {bot_response}")
        step += 1

# Start the chatbot
start_chatbot()

Loading the transformer model... (This might take a minute)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!

--------------------------------------------------
Chatbot: Hello! I am your AI assistant. How can I help you today?
(Type 'exit' or 'quit' to stop the conversation)
--------------------------------------------------
User: Hello!


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: Hi! : 3
User: What is AI?
Chatbot: AI is a type of AI.
User: bye
Chatbot: Bye bye!
User: exit
Chatbot: Goodbye! Have a great day!
